# Hamilton PSD/4 — usage

How to drive the pump. Nine cells, all of them things you'd actually write in
a protocol.

**This moves a syringe and a valve.** Have the fluid path connected to
something you don't mind, and start Jupyter with `sudo` — the serial port is
root-owned:

```bash
cd /home/dorna/Downloads/workspace/workspace
sudo jupyter lab --allow-root
```


## Connect

One call: opens the port, configures the pump, and homes it.

Two things MUST be declared because the pump cannot report them, and both
scale every volume silently if wrong: `variant` (the step scale —
`'standard'` = PSD/4, PSD/6 high-torque; `'smooth_flow'` = the SF drives)
and `syringe_volume_ul` (what the 30 mm stroke sweeps of the fitted barrel).
Valve type is NOT declared — DIP switches 4-6 own it and the pump reports it.


In [ ]:
import sys, time
sys.path.insert(0, '/home/dorna/Downloads/workspace/workspace')
from workspace.components.pump.psd4_station import PSD4Station

pump = PSD4Station(
    port='/dev/serial/by-id/usb-FTDI_FT232R_USB_UART_BG01X3BN-if00-port0',
    address=1,                # rotary switch on the back
    syringe_volume_ul=100.0,  # what the 30 mm stroke SWEEPS of the fitted barrel
    variant='smooth_flow',    # 'standard' (PSD/4, PSD/6) or 'smooth_flow' (SF drives)
    high_resolution=False,    # standard = 8x faster strokes; high-res only buys granularity
    simulation=False,         # valve type comes from DIP switches 4-6, nothing to declare
)

print('connected:', pump.recover(), '|', pump.state, pump.msg)
print('valve type:', pump.valve_type)


## Initialize

Homes the plunger and valve. Needed after every power-up, and after any stop.
`recover()` above already did it — this is how you'd redo it, or switch to a
different barrel.


In [ ]:
pump.initialize()   # homes, and re-applies the speed (default 100)
pump.set_speed(100)
print('holding', pump.volume(), 'uL')


## Aspirate and dispense

The port is an argument — the valve moves there, then the plunger does its
thing. Ports are numbered 1..N going round the valve.

Draw 80 µL from port 1, then split it: 20 µL to port 2, 60 µL to port 3.


In [ ]:
pump.aspirate(80, port=1)
print('after aspirate:', pump.volume(), 'uL')

pump.dispense(20, port=2)
print('after 20 to p2:', pump.volume(), 'uL')

pump.dispense(60, port=3)
print('after 60 to p3:', pump.volume(), 'uL')


## Several draws, one dispense

The other way round — combine from two sources and push the lot out at once.
Leave `port` off and the move uses whatever port the valve is already on.


In [ ]:
pump.aspirate(60, port=1)
pump.aspirate(40, port=2)
print('holding:', pump.volume(), 'uL')

pump.dispense(100, port=3)
print('after  :', pump.volume(), 'uL')


## Absolute moves

`aspirate`/`dispense` are relative — "50 more", "20 less". `move_to_volume`
is absolute — "end up holding exactly this". Use it when you don't know where
you are, e.g. first thing in a protocol.

`empty()` is `move_to_volume(0)`, named for the common case.


In [ ]:
pump.aspirate(90, port=1)
print('holding      :', pump.volume(), 'uL')

pump.move_to_volume(30, port=3)      # pushes 60 out of port 3
print('after move_to:', pump.volume(), 'uL')

pump.empty(port=3)
print('after empty  :', pump.volume(), 'uL')


## Speed

0-100, normalized across pump variants and resolution modes: 100 is the
fastest preset (~2.3 s for a full stroke on this pump in standard
resolution), 0 the slowest (~600 s). The preset ladder is roughly
geometric, so mid-scale sits much closer to the slow end in seconds.
The speed survives initialize — the station re-applies the last one set.


In [ ]:
for pct in (100, 80):
    pump.set_speed(pct)
    t = time.time()
    pump.aspirate(100, port=1)
    print(f'speed {pct:3}/100 -> 100 uL took {time.time()-t:.1f}s')
    pump.empty(port=3)

pump.set_speed(100)


## Priming

`prime(n)` runs n full-barrel fill/empty cycles to flush air out of the
fluid path — 0-100 on a 100 µL barrel, 0-250 on a 250, whatever is
declared. Absolute moves, so it works from whatever volume you happen to
be holding.


In [ ]:
pump.prime(2, from_port=1, to_port=3)

print('primed, holding:', pump.volume(), 'uL')


## Asking for too much

Ops return `True`/`False` rather than raising, so a protocol can just check.
The barrel limit is enforced here — the pump itself does not check it.


In [ ]:
pump.aspirate(100, port=1)
print('full        :', pump.volume(), 'uL')
print('50 more     :', pump.aspirate(50), '  <- refused, barrel is full')
print('still       :', pump.volume(), 'uL')

pump.empty(port=3)
print('150 out     :', pump.dispense(150), '  <- refused, nothing to give')
print('status      :', pump.status())


## Done

`release()` closes the port cleanly, so unplugging doesn't raise an alarm.


In [ ]:
pump.empty(port=3)
pump.valve(1)
pump.release()
print('released:', pump.state)
